# Notebook 3 — Flow-Based Feature Engineering

Raw Wireshark packets are not suitable for ML directly — a single attack can look normal at the
packet level. Instead, we group packets into **flows** (same source, destination, protocol within
a time window) and extract **statistical features** over each flow.

This makes the model generalise to real-world captures where background traffic is present.

**All feature extraction logic lives in `utils/feature_engineering.py`** so that Notebook 5
(inference) uses the exact same code — preventing train/inference skew.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure utils/ is importable regardless of working directory
sys.path.insert(0, str(Path('.').resolve()))

from utils.feature_engineering import (
    preprocess_packets,
    extract_flow_features,
    FEATURE_COLS,
    TIME_WINDOW_SECONDS,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

DATA_DIR = Path('data')
print(f'Time window for flow grouping: {TIME_WINDOW_SECONDS}s')
print(f'Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')

## 1. Load raw combined data

In [ ]:
raw = pd.read_parquet(DATA_DIR / 'raw_combined.parquet')
print(f'Raw data: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
print('Labels:', raw['label'].value_counts().to_dict())

## 2. Preprocess packets

Clean types, parse TCP flags and HTTP keywords from the `Info` column.

In [ ]:
print('Preprocessing packets...')
preprocessed = preprocess_packets(raw)

print(f'After preprocessing: {preprocessed.shape[0]:,} rows x {preprocessed.shape[1]} columns')
print('\nSample of parsed flag columns:')
flag_cols = ['info', 'syn', 'ack', 'fin', 'rst', 'http_req', 'http_resp', 'dst_port']
preprocessed[flag_cols].head(10)

In [ ]:
# Quick check: flag counts per class
flag_summary = preprocessed.groupby('label')[['syn','ack','fin','rst','http_req','http_resp']].sum()
print('TCP/HTTP flag totals per class:')
print(flag_summary.to_string())

## 3. Extract flow-level features

Group packets into flows using a `5-second` sliding time window keyed on
`(Source, Destination, Protocol)`. Then compute statistical aggregates over each flow.

In [ ]:
print('Extracting flow features — this may take a few minutes on large datasets...')

# Process each source file separately to avoid cross-capture flow leakage
flow_dfs = []

for src_file, grp in preprocessed.groupby('src_file'):
    label = grp['label'].iloc[0]
    flows = extract_flow_features(grp, label_col='label')
    flows['src_file'] = src_file
    flow_dfs.append(flows)
    print(f'  {src_file:60s}  {len(grp):>7,} pkts → {len(flows):>5,} flows  [{label}]')

features = pd.concat(flow_dfs, ignore_index=True)
print(f'\nTotal flows extracted: {len(features):,}')

## 4. Validate feature DataFrame

In [ ]:
print('Feature DataFrame shape:', features.shape)
print('\nColumns:', features.columns.tolist())
print('\nLabel distribution (flows):')
print(features['label'].value_counts().to_string())

In [ ]:
print('\nMissing values in feature columns:')
missing = features[FEATURE_COLS].isnull().sum()
print(missing[missing > 0] if missing.any() else 'None — all clean!')

print('\nDescriptive statistics:')
features[FEATURE_COLS].describe().round(3)

## 5. Visualise flow-level features

In [ ]:
# Flow count per class
flow_counts = features['label'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
flow_counts.plot(kind='bar', ax=ax, color=sns.color_palette('tab10', len(flow_counts)))
ax.set_title('Flow count per class', fontsize=13)
ax.set_xlabel(''); ax.set_ylabel('Number of flows')
ax.tick_params(axis='x', rotation=30)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(DATA_DIR / 'plots' / '08_flow_count_per_class.png', dpi=150)
plt.show()

In [ ]:
# Key feature distributions per class
key_features = ['packet_count', 'avg_length', 'avg_iat', 'packets_per_second', 'syn_count']

fig, axes = plt.subplots(1, len(key_features), figsize=(18, 4))

for ax, feat in zip(axes, key_features):
    sns.boxplot(data=features, x='label', y=feat,
                palette='tab10', showfliers=False, ax=ax)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=40)

plt.suptitle('Flow-level feature distributions per class', fontsize=13)
plt.tight_layout()
plt.savefig(DATA_DIR / 'plots' / '09_flow_feature_boxplots.png', dpi=150)
plt.show()

## 6. Save features to Parquet

In [ ]:
out_path = DATA_DIR / 'features.parquet'
features.to_parquet(out_path, index=False)
size_mb = out_path.stat().st_size / 1e6
print(f'Saved {len(features):,} flows to {out_path}  ({size_mb:.2f} MB)')
print(f'Columns saved: {features.columns.tolist()}')

## ✅ Notebook 3 complete
**Output:** `data/features.parquet`  
Each row = one network flow with 18 statistical features + label.  
**Next:** Run `04_model_training.ipynb`